
# Kuroshio Daily 1993–2025 — 数据构建 + 自动流轴追踪
## CMEMS 1993–2024 multiyear + 既有 2023–2025 daily 数据

这个 Notebook 为正式重训准备一套尽量同质、可追溯、可断点恢复的长时段日尺度黑潮路径数据。

### 数据源

**主历史源（优先）**

```text
Windows:
F:\01-CMEMS\GLOBAL_MULTIYEAR_PHY_001_030\
cmems_mod_glo_phy_myint_0.083deg_P1D-m\1993 ... 2024

WSL:
/mnt/f/01-CMEMS/GLOBAL_MULTIYEAR_PHY_001_030/
cmems_mod_glo_phy_myint_0.083deg_P1D-m/1993 ... 2024
```

**既有近期源**

```text
E:\2023
E:\2024
E:\2025

WSL:
/mnt/e/2023
/mnt/e/2024
/mnt/e/2025
```

### 合并原则

为了让长期训练序列尽可能同质：

```text
1993–2024:
优先使用新的 multiyear 产品

若 multiyear 某一天缺失、而旧近期源在2023/2024有同日数据：
允许 recent fallback

2025:
使用原 2023–2025 数据源
```

2023–2024 的重叠文件不会重复进入训练集。

### 相较旧数据构建脚本的优化

1. 支持 **“一日一个NetCDF”与“一个NetCDF含多日time records”**；
2. NetCDF 只读取研究区域与 50 m 附近两个深度层，不再把完整三维场读入内存；
3. 50 m 采用两深度层线性插值；
4. **不再为全部约1.2万天保存50 m缓存**：成功日只保存流轴；仅人工复核日保存50 m场；
5. 完整支持断点续跑：每一天独立保存 track + metadata；
6. 保留原 characteristic-line 方法 + failure-only auto-retry，不再使用 CNN；
7. 完成后进行相邻日连续性审计，将“成功但可疑”的路径也送入人工标注队列；
8. 人工标注仍使用之前的鼠标交互逻辑，另见配套 ManualQC Notebook。

> 这一阶段主要受 F/E 盘 I/O 和 CPU 影响，RTX 5080 基本不参与。


In [ ]:

# ===== 0. 配置 =====
from pathlib import Path
from dataclasses import dataclass, asdict
import os, re, json, math, random, warnings, shutil, gc, time
import numpy as np
import pandas as pd

@dataclass
class CFG:
    PROJECT_ROOT: str = "/mnt/d/KuroshioDaily1993_2025"

    # 新的 1993–2024 CMEMS multiyear 数据
    MULTIYEAR_ROOT: str = (
        "/mnt/f/01-CMEMS/GLOBAL_MULTIYEAR_PHY_001_030/"
        "cmems_mod_glo_phy_myint_0.083deg_P1D-m"
    )
    MULTIYEAR_START_YEAR: int = 1993
    MULTIYEAR_END_YEAR: int = 2024

    # 之前的 2023–2025 数据
    RECENT_ROOT: str = "/mnt/e"
    RECENT_YEARS: tuple = (2023, 2024, 2025)

    START_DATE: str = "1993-01-01"
    END_DATE: str | None = None  # None = 自动采用扫描到的最后一天

    NETCDF_SUFFIXES: tuple = (".nc", ".nc4", ".cdf")

    # 黑潮研究区域；比实际绘图范围略留边界
    LON_MIN: float = 119.0
    LON_MAX: float = 150.0
    LAT_MIN: float = 19.0
    LAT_MAX: float = 43.0
    TARGET_DEPTH_M: float = 50.0

    # 一文件多time时必须匹配对应日期
    TIME_MATCH_TOLERANCE_HOURS: float = 18.0

    # source merge
    PREFER_MULTIYEAR_THROUGH: str = "2024-12-31"
    OVERLAP_AUDIT_DAYS: int = 18
    STRICT_OVERLAP_AUDIT: bool = False
    OVERLAP_WARN_SPEED_RMSE_MS: float = 0.15
    OVERLAP_WARN_CORRELATION: float = 0.95

    # characteristic-line
    SEED_LON_MIN: float = 121.0
    SEED_LON_MAX: float = 122.0
    SEED_LAT_MIN: float = 22.0
    SEED_LAT_MAX: float = 23.0
    TERMINAL_LON: float = 146.0
    EXTENSION_LON: float = 141.0
    AUX_HALF_KM: float = 70.0
    AUX_POINTS: int = 21
    STEP_MAIN_KM: float = 10.0
    STEP_EXTENSION_KM: float = 50.0
    MAX_TRACK_POINTS: int = 1200

    # 原始FIX2参数
    BASE_RESCUE_JUMP_KM: float = 20.0
    BASE_RESCUE_ANGLE_DEG: float = 30.0
    BASE_LOOP_LIMIT: int = 3
    BASE_REPEAT_THRESHOLD_KM: float = 1.5
    RESCUE_STEPS: int = 20

    # failure-only retry：为了32年批处理缩减为6组高价值组合
    RETRY_GRID: tuple = (
        (20.0, 45.0, 4, 1.5),
        (30.0, 30.0, 4, 1.5),
        (30.0, 45.0, 4, 1.5),
        (40.0, 45.0, 4, 1.5),
        (40.0, 60.0, 5, 2.5),
        (50.0, 60.0, 5, 2.5),
    )
    MAX_GOOD_RETRY_CANDIDATES: int = 3

    # 自动物理QC：故意比之前稍宽，避免把大量真实大蛇行送去人工
    RIDGE_HALF_KM: float = 35.0
    RIDGE_POINTS: int = 11
    RIDGE_SAMPLE_STRIDE: int = 5
    MIN_RIDGE_MEDIAN: float = 0.85
    MIN_RIDGE_P10: float = 0.60
    MAX_RIDGE_LOW_FRAC: float = 0.25
    MAX_P95_TURN_DEG: float = 145.0

    # 相邻日连续性只用于“可疑日筛查”
    TEMPORAL_MAX_GAP_DAYS: int = 3
    TEMPORAL_WARN_KM_1D: float = 100.0
    TEMPORAL_WARN_KM_3D: float = 180.0

    # 批处理
    FORCE_REPROCESS: bool = False
    SAVE_FIELD_FOR_ALL_DAYS: bool = False  # 强烈建议False
    SAVE_FIELD_FOR_MANUAL_ONLY: bool = True
    FLUSH_REPORT_EVERY: int = 50
    SMOKE_TEST_ONLY: bool = False
    SMOKE_DATES: tuple = (
        "1993-01-15",
        "2000-07-15",
        "2010-01-15",
        "2020-07-15",
        "2024-07-15",
        "2025-02-15",
    )

    RANDOM_SEED: int = 20260930

cfg = CFG()

ROOT = Path(cfg.PROJECT_ROOT)
MY_ROOT = Path(cfg.MULTIYEAR_ROOT)
RECENT_ROOT = Path(cfg.RECENT_ROOT)

DIRS = {
    "reports": ROOT/"reports",
    "tracks_auto": ROOT/"tracks_auto",
    "track_meta": ROOT/"track_meta",
    "manual_cache": ROOT/"manual_cache_50m",
    "dataset": ROOT/"dataset",
    "figures": ROOT/"figures",
}
for p in [ROOT,*DIRS.values()]:
    p.mkdir(parents=True,exist_ok=True)

random.seed(cfg.RANDOM_SEED)
np.random.seed(cfg.RANDOM_SEED)

print(json.dumps(asdict(cfg),indent=2,ensure_ascii=False))

for p,label in [(MY_ROOT,"MULTIYEAR_ROOT"),(RECENT_ROOT,"RECENT_ROOT")]:
    print(label,p,"exists=",p.exists())
    if not p.exists():
        raise FileNotFoundError(
            f"{label} 不存在: {p}\n"
            "Windows盘通常应自动挂载为 /mnt/f 或 /mnt/e。"
        )



## WSL 目录快速检查

如果上一个单元提示 `/mnt/f` 或 `/mnt/e` 不存在，可在 WSL 终端检查：

```bash
ls /mnt/f
ls /mnt/e
```

若 F 盘未自动挂载：

```bash
sudo mkdir -p /mnt/f
sudo mount -t drvfs F: /mnt/f
```

E 盘同理：

```bash
sudo mkdir -p /mnt/e
sudo mount -t drvfs E: /mnt/e
```


In [ ]:

# ===== 1. 环境预检 =====
import importlib.util, sys, platform, subprocess

required=["xarray","scipy","tqdm"]
missing=[x for x in required if importlib.util.find_spec(x) is None]
if missing:
    raise RuntimeError(
        "缺少依赖: "+", ".join(missing)+"\n"
        "在WSL虚拟环境执行:\n"
        "python -m pip install -U xarray netcdf4 h5netcdf scipy tqdm pandas"
    )

import xarray as xr
from tqdm.auto import tqdm
from scipy.interpolate import RegularGridInterpolator

print("Python:",sys.version.split()[0])
print("platform:",platform.platform())
print("xarray:",xr.__version__)


In [ ]:

# ===== 2. 扫描两套数据源，并支持multi-time NetCDF =====
DATE8 = re.compile(r"(?<!\d)((?:19|20)\d{6})(?!\d)")
REV_PATTERN = re.compile(r"(?:_R|-R|R)(\d{8})", re.I)

def valid_date_tokens(name):
    out=[]
    for s in DATE8.findall(name):
        try:
            out.append(pd.to_datetime(s,format="%Y%m%d").normalize())
        except Exception:
            pass
    return sorted(set(out))

def revision_from_filename(name):
    m=REV_PATTERN.search(name)
    if not m:
        return pd.NaT
    try:
        return pd.to_datetime(m.group(1),format="%Y%m%d").normalize()
    except Exception:
        return pd.NaT

def netcdf_time_records(path):
    """只读取time坐标；用于日期范围文件或无法从文件名唯一解析日期的文件。"""
    rows=[]
    try:
        with xr.open_dataset(path,decode_times=True,cache=False) as ds:
            tname=next(
                (x for x in ["time","TIME","time_counter"]
                 if x in ds.coords or x in ds.variables or x in ds.dims),
                None
            )
            if tname is None:
                return rows
            vals=pd.to_datetime(np.asarray(ds[tname].values).ravel(),errors="coerce")
            for i,t in enumerate(vals):
                if pd.isna(t):
                    continue
                rows.append((pd.Timestamp(t).normalize(),int(i)))
    except Exception:
        pass
    return rows

def scan_source(root,year_dirs,source_name):
    files=[]
    for year in year_dirs:
        yd=root/str(year)
        if not yd.exists():
            print("[WARN] missing year dir:",yd)
            continue
        files.extend([
            p for p in yd.rglob("*")
            if p.is_file() and p.suffix.lower() in cfg.NETCDF_SUFFIXES
        ])

    files=sorted(files)
    print(source_name,"NetCDF files:",len(files))

    rows=[]
    unparsed=[]

    for p in tqdm(files,desc=f"scan {source_name}"):
        tokens=valid_date_tokens(p.name)
        # revision token不作为观测日期候选
        rev=revision_from_filename(p.name)
        if not pd.isna(rev):
            tokens=[t for t in tokens if t!=rev]

        # 唯一8位日期时走快速路径；多日期/无日期时读取time坐标展开
        records=[]
        if len(tokens)==1:
            records=[(tokens[0],None,"filename")]
        else:
            trs=netcdf_time_records(p)
            if trs:
                records=[(d,ti,"netcdf_time") for d,ti in trs]

        st=p.stat()
        if not records:
            unparsed.append({
                "source":source_name,
                "path":str(p),
                "filename":p.name,
                "size_bytes":int(st.st_size)
            })
            continue

        for dt,ti,method in records:
            rows.append({
                "date":dt,
                "source":source_name,
                "path":str(p),
                "time_index":ti,
                "filename":p.name,
                "parse_method":method,
                "revision_date":rev,
                "size_bytes":int(st.st_size),
                "mtime_ns":int(st.st_mtime_ns),
            })

    return pd.DataFrame(rows),pd.DataFrame(unparsed)

my_years=range(cfg.MULTIYEAR_START_YEAR,cfg.MULTIYEAR_END_YEAR+1)
recent_years=cfg.RECENT_YEARS

my_scan,my_unparsed=scan_source(
    MY_ROOT,my_years,"MULTIYEAR"
)
recent_scan,recent_unparsed=scan_source(
    RECENT_ROOT,recent_years,"RECENT"
)

all_scan=pd.concat([my_scan,recent_scan],ignore_index=True)
all_unparsed=pd.concat([my_unparsed,recent_unparsed],ignore_index=True)

all_scan.to_csv(
    DIRS["reports"]/"00_全部NetCDF日期记录.csv",
    index=False,encoding="utf-8-sig"
)
all_unparsed.to_csv(
    DIRS["reports"]/"00_无法解析日期的NetCDF.csv",
    index=False,encoding="utf-8-sig"
)

print("date records:",len(all_scan))
print("unparsed files:",len(all_unparsed))


In [ ]:

# ===== 3. 每个source内部去重，然后合并1993–2025 manifest =====
def choose_one_per_source(scan):
    if len(scan)==0:
        return scan.copy()

    q=scan.copy()
    q["revision_rank"]=q["revision_date"].fillna(pd.Timestamp("1900-01-01"))
    q=q.sort_values(
        ["date","revision_rank","size_bytes","mtime_ns","filename"],
        ascending=[True,False,False,False,False]
    )
    return q.drop_duplicates("date",keep="first")

my_one=choose_one_per_source(my_scan)
recent_one=choose_one_per_source(recent_scan)

my_cols={
    "path":"my_path","time_index":"my_time_index",
    "filename":"my_filename","size_bytes":"my_size_bytes"
}
recent_cols={
    "path":"recent_path","time_index":"recent_time_index",
    "filename":"recent_filename","size_bytes":"recent_size_bytes"
}

m=my_one[["date","path","time_index","filename","size_bytes"]].rename(columns=my_cols)
r=recent_one[["date","path","time_index","filename","size_bytes"]].rename(columns=recent_cols)

merged=m.merge(r,on="date",how="outer").sort_values("date")

if len(merged)==0:
    raise RuntimeError("没有扫描到任何可用日期。")

date0=max(pd.Timestamp(cfg.START_DATE),merged["date"].min())
date1=(
    pd.Timestamp(cfg.END_DATE)
    if cfg.END_DATE is not None
    else merged["date"].max()
)
prefer_end=pd.Timestamp(cfg.PREFER_MULTIYEAR_THROUGH)

expected=pd.DataFrame({"date":pd.date_range(date0,date1,freq="D")})
manifest=expected.merge(merged,on="date",how="left")

def select_source(row):
    dt=row["date"]
    has_my=pd.notna(row["my_path"])
    has_recent=pd.notna(row["recent_path"])

    if dt<=prefer_end:
        if has_my:
            return "MULTIYEAR",row["my_path"],row["my_time_index"]
        if has_recent:
            return "RECENT_FALLBACK",row["recent_path"],row["recent_time_index"]
    else:
        if has_recent:
            return "RECENT",row["recent_path"],row["recent_time_index"]
        if has_my:
            return "MULTIYEAR_FALLBACK",row["my_path"],row["my_time_index"]

    return "MISSING",None,None

sel=manifest.apply(select_source,axis=1,result_type="expand")
sel.columns=["selected_source","selected_path","selected_time_index"]
manifest=pd.concat([manifest,sel],axis=1)
manifest["raw_status"]=np.where(
    manifest["selected_path"].notna(),"present","missing"
)
manifest["has_overlap"]=(
    manifest["my_path"].notna()&
    manifest["recent_path"].notna()
)

manifest_path=DIRS["reports"]/"01_最终源合并Manifest.csv"
manifest.to_csv(
    manifest_path,index=False,encoding="utf-8-sig"
)

annual=(
    manifest.assign(year=manifest["date"].dt.year)
    .groupby("year",as_index=False)
    .agg(
        theoretical_days=("date","size"),
        present_days=("raw_status",lambda x:int((x=="present").sum())),
        missing_days=("raw_status",lambda x:int((x=="missing").sum())),
        multiyear_days=("selected_source",lambda x:int(x.astype(str).str.startswith("MULTIYEAR").sum())),
        recent_days=("selected_source",lambda x:int(x.astype(str).str.startswith("RECENT").sum())),
    )
)
annual.to_csv(
    DIRS["reports"]/"01_年度数据覆盖统计.csv",
    index=False,encoding="utf-8-sig"
)

print("date range:",date0.date(),"->",date1.date())
print("theoretical days:",len(manifest))
print("present:",int((manifest.raw_status=="present").sum()))
print("missing:",int((manifest.raw_status=="missing").sum()))
print("overlap 2023/24 days:",int(manifest.has_overlap.sum()))
display(annual.tail(10))


In [ ]:

# ===== 4. 高效50m读取：只读ROI + 两个相邻深度层 =====
NAME_CANDIDATES={
    "u":["uo","u","eastward_sea_water_velocity"],
    "v":["vo","v","northward_sea_water_velocity"],
    "lon":["longitude","lon","x"],
    "lat":["latitude","lat","y"],
    "depth":["depth","deptht","lev","z"],
    "time":["time","TIME","time_counter"],
}

def pick_name(ds,candidates,kind):
    names=list(dict.fromkeys(
        list(ds.variables)+list(ds.coords)+list(ds.dims)
    ))
    for c in candidates:
        if c in names:
            return c
    raise KeyError(
        f"无法找到 {kind}; candidates={candidates}; names={names[:80]}"
    )

def index_bounds(coord,lo,hi):
    arr=np.asarray(coord,float)
    ids=np.where((arr>=lo)&(arr<=hi))[0]
    if len(ids)==0:
        raise RuntimeError(f"坐标裁剪为空: {lo}~{hi}; actual={arr.min()}~{arr.max()}")
    return ids

def choose_time_index(ds,tname,target_date,preferred_index):
    if tname is None:
        return None

    nt=int(ds.sizes.get(tname,1))
    if nt<=1:
        return 0

    if preferred_index is not None and not pd.isna(preferred_index):
        ii=int(preferred_index)
        if 0<=ii<nt:
            return ii

    vals=pd.to_datetime(
        np.asarray(ds[tname].values).ravel(),
        errors="coerce"
    )
    target=pd.Timestamp(target_date)
    dh=np.array([
        abs((pd.Timestamp(x)-target).total_seconds())/3600
        if not pd.isna(x) else np.inf
        for x in vals
    ])
    k=int(np.argmin(dh))
    if dh[k]>cfg.TIME_MATCH_TOLERANCE_HOURS:
        raise RuntimeError(
            f"time coordinate 无法匹配 {target_date}; nearest diff={dh[k]:.1f} h"
        )
    return k

def read_50m_field(nc_path,target_date,time_index=None):
    nc_path=Path(nc_path)

    with xr.open_dataset(
        nc_path,
        decode_times=True,
        cache=False
    ) as ds:

        un=pick_name(ds,NAME_CANDIDATES["u"],"u")
        vn=pick_name(ds,NAME_CANDIDATES["v"],"v")
        xn=pick_name(ds,NAME_CANDIDATES["lon"],"lon")
        yn=pick_name(ds,NAME_CANDIDATES["lat"],"lat")
        zn=pick_name(ds,NAME_CANDIDATES["depth"],"depth")
        tn=next(
            (x for x in NAME_CANDIDATES["time"]
             if x in ds.dims or x in ds.coords),
            None
        )

        lon=np.asarray(ds[xn].values,float)
        lat=np.asarray(ds[yn].values,float)
        dep=np.asarray(ds[zn].values,float)

        xids=index_bounds(lon,cfg.LON_MIN,cfg.LON_MAX)
        yids=index_bounds(lat,cfg.LAT_MIN,cfg.LAT_MAX)

        dep_order=np.argsort(dep)
        dep_sorted=dep[dep_order]

        if not (
            np.nanmin(dep_sorted)<=cfg.TARGET_DEPTH_M<=np.nanmax(dep_sorted)
        ):
            raise RuntimeError(
                f"{nc_path.name}: depth {dep_sorted.min()}~{dep_sorted.max()} "
                f"不包含 {cfg.TARGET_DEPTH_M}m"
            )

        k_hi=int(np.searchsorted(dep_sorted,cfg.TARGET_DEPTH_M,side="left"))
        if k_hi==0:
            k_lo=k_hi=0
        elif k_hi>=len(dep_sorted):
            k_lo=k_hi=len(dep_sorted)-1
        elif np.isclose(dep_sorted[k_hi],cfg.TARGET_DEPTH_M):
            k_lo=k_hi
        else:
            k_lo=k_hi-1

        iz0=int(dep_order[k_lo])
        iz1=int(dep_order[k_hi])

        it=choose_time_index(
            ds,tn,target_date,time_index
        )

        def load_level(varname,iz):
            da=ds[varname]
            indexers={}
            if tn is not None and tn in da.dims:
                indexers[tn]=it
            if zn in da.dims:
                indexers[zn]=iz
            if yn in da.dims:
                indexers[yn]=yids
            if xn in da.dims:
                indexers[xn]=xids

            da=da.isel(indexers).squeeze(drop=True)

            if yn not in da.dims or xn not in da.dims:
                raise RuntimeError(
                    f"{varname} after isel dims={da.dims}; expected {yn},{xn}"
                )
            da=da.transpose(yn,xn)
            return np.asarray(da.load().values,dtype=np.float32)

        u0=load_level(un,iz0)
        v0=load_level(vn,iz0)

        d0=float(dep[iz0])

        if iz1==iz0:
            u=u0
            v=v0
            d1=d0
        else:
            u1=load_level(un,iz1)
            v1=load_level(vn,iz1)
            d1=float(dep[iz1])

            w=(cfg.TARGET_DEPTH_M-d0)/(d1-d0)
            u=((1-w)*u0+w*u1).astype(np.float32)
            v=((1-w)*v0+w*v1).astype(np.float32)

        lon2=lon[xids]
        lat2=lat[yids]

        # 统一为经纬度升序
        if lon2[0]>lon2[-1]:
            lon2=lon2[::-1]
            u=u[:,::-1]
            v=v[:,::-1]
        if lat2[0]>lat2[-1]:
            lat2=lat2[::-1]
            u=u[::-1,:]
            v=v[::-1,:]

        finite=np.isfinite(u)&np.isfinite(v)
        if finite.mean()<0.08:
            raise RuntimeError(
                f"{nc_path.name}: ROI有效流速格点比例过低 {finite.mean():.3f}"
            )

        return {
            "lon":lon2.astype(np.float64),
            "lat":lat2.astype(np.float64),
            "u":u.astype(np.float32),
            "v":v.astype(np.float32),
            "depth_lo_m":d0,
            "depth_hi_m":d1,
            "source_file":str(nc_path),
        }

def save_field_npz(field,path):
    path=Path(path)
    path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(".tmp.npz")
    np.savez_compressed(
        tmp,
        lon=field["lon"],lat=field["lat"],
        u=field["u"],v=field["v"],
        depth_lo_m=np.array(field["depth_lo_m"],np.float32),
        depth_hi_m=np.array(field["depth_hi_m"],np.float32),
        source_file=np.array(field["source_file"]),
        target_depth_m=np.array(cfg.TARGET_DEPTH_M,np.float32),
    )
    os.replace(tmp,path)

# 多年代结构smoke audit
audit_dates=[]
for yr in [1993,2000,2010,2020,2024,2025]:
    q=manifest[
        (manifest.raw_status=="present")&
        (manifest.date.dt.year==yr)
    ]
    if len(q):
        audit_dates.append(pd.Timestamp(q.iloc[len(q)//2].date))

audit=[]
for dt in audit_dates:
    r=manifest[manifest.date==dt].iloc[0]
    try:
        f=read_50m_field(
            r.selected_path,dt,r.selected_time_index
        )
        audit.append({
            "date":dt,
            "source":r.selected_source,
            "ok":True,
            "lon_n":len(f["lon"]),
            "lat_n":len(f["lat"]),
            "finite_frac":float(
                np.mean(np.isfinite(f["u"])&np.isfinite(f["v"]))
            ),
            "depth_lo_m":f["depth_lo_m"],
            "depth_hi_m":f["depth_hi_m"],
        })
    except Exception as e:
        audit.append({
            "date":dt,
            "source":r.selected_source,
            "ok":False,
            "error":repr(e)
        })

audit_df=pd.DataFrame(audit)
audit_df.to_csv(
    DIRS["reports"]/"02_跨年代NetCDF结构与50m读取审计.csv",
    index=False,encoding="utf-8-sig"
)
display(audit_df)

if not audit_df["ok"].all():
    raise RuntimeError(
        "跨年代50m读取预审计失败。"
        "先检查 02_跨年代NetCDF结构与50m读取审计.csv。"
    )


In [ ]:

# ===== 5. 2023–2024 两套源重叠一致性抽样审计 =====
overlap=manifest[
    manifest["has_overlap"]&
    (manifest["date"]<=pd.Timestamp("2024-12-31"))
].copy()

audit_rows=[]

if len(overlap):
    ids=np.unique(
        np.linspace(
            0,len(overlap)-1,
            min(cfg.OVERLAP_AUDIT_DAYS,len(overlap)),
            dtype=int
        )
    )

    for ix in tqdm(ids,desc="overlap audit"):
        r=overlap.iloc[ix]
        dt=pd.Timestamp(r.date)

        try:
            a=read_50m_field(
                r.my_path,dt,r.my_time_index
            )
            b=read_50m_field(
                r.recent_path,dt,r.recent_time_index
            )

            # 将recent插值到multiyear网格
            bu=RegularGridInterpolator(
                (b["lat"],b["lon"]),b["u"],
                bounds_error=False,fill_value=np.nan
            )
            bv=RegularGridInterpolator(
                (b["lat"],b["lon"]),b["v"],
                bounds_error=False,fill_value=np.nan
            )

            LLON,LLAT=np.meshgrid(a["lon"],a["lat"])
            pts=np.column_stack([LLAT.ravel(),LLON.ravel()])
            ub=bu(pts).reshape(LLON.shape)
            vb=bv(pts).reshape(LLON.shape)

            valid=(
                np.isfinite(a["u"])&
                np.isfinite(a["v"])&
                np.isfinite(ub)&
                np.isfinite(vb)
            )

            spa=np.hypot(a["u"],a["v"])
            spb=np.hypot(ub,vb)

            corr=np.corrcoef(
                spa[valid].ravel(),
                spb[valid].ravel()
            )[0,1]

            audit_rows.append({
                "date":dt,
                "u_rmse_ms":float(np.sqrt(np.mean((a["u"][valid]-ub[valid])**2))),
                "v_rmse_ms":float(np.sqrt(np.mean((a["v"][valid]-vb[valid])**2))),
                "speed_rmse_ms":float(np.sqrt(np.mean((spa[valid]-spb[valid])**2))),
                "speed_corr":float(corr),
                "n_valid":int(valid.sum()),
            })

        except Exception as e:
            audit_rows.append({
                "date":dt,
                "error":repr(e)
            })

overlap_audit=pd.DataFrame(audit_rows)
overlap_audit.to_csv(
    DIRS["reports"]/"03_2023_2024两套源重叠场一致性审计.csv",
    index=False,encoding="utf-8-sig"
)

if len(overlap_audit):
    display(overlap_audit)
    good=overlap_audit.dropna(subset=["speed_rmse_ms","speed_corr"])

    if len(good):
        warn=(
            good["speed_rmse_ms"].median()>cfg.OVERLAP_WARN_SPEED_RMSE_MS
            or good["speed_corr"].median()<cfg.OVERLAP_WARN_CORRELATION
        )
        print(
            "median speed RMSE:",
            good["speed_rmse_ms"].median(),
            "median corr:",
            good["speed_corr"].median()
        )
        if warn:
            msg=(
                "两套源在重叠期存在明显差异。"
                "脚本仍按设计优先MULTIYEAR到2024年底、RECENT从2025开始；"
                "后续应重点检查2024/2025产品切换连续性。"
            )
            if cfg.STRICT_OVERLAP_AUDIT:
                raise RuntimeError(msg)
            warnings.warn(msg)


In [ ]:

# ===== 6. characteristic-line 几何与tracker =====
EARTH_R_KM=6371.0088

def wrap_pi(a):
    return (a+np.pi)%(2*np.pi)-np.pi

def haversine_km(lon1,lat1,lon2,lat2):
    lon1,lat1,lon2,lat2=map(np.asarray,(lon1,lat1,lon2,lat2))
    p1,p2=np.deg2rad(lat1),np.deg2rad(lat2)
    dl=np.deg2rad(lon2-lon1)
    dp=p2-p1
    aa=np.sin(dp/2)**2+np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2*EARTH_R_KM*np.arcsin(np.minimum(1.0,np.sqrt(aa)))

def angle_east_rad(lon1,lat1,lon2,lat2):
    p1,p2=np.deg2rad(lat1),np.deg2rad(lat2)
    dl=np.deg2rad(lon2-lon1)
    y=np.sin(dl)*np.cos(p2)
    x=np.cos(p1)*np.sin(p2)-np.sin(p1)*np.cos(p2)*np.cos(dl)
    nav=np.arctan2(y,x)
    return wrap_pi(np.pi/2-nav)

def forward_point(lon,lat,east_angle,distance_km):
    nav=np.pi/2-east_angle
    phi1,lam1=np.deg2rad(lat),np.deg2rad(lon)
    d=distance_km/EARTH_R_KM

    phi2=np.arcsin(
        np.sin(phi1)*np.cos(d)+
        np.cos(phi1)*np.sin(d)*np.cos(nav)
    )
    lam2=lam1+np.arctan2(
        np.sin(nav)*np.sin(d)*np.cos(phi1),
        np.cos(d)-np.sin(phi1)*np.sin(phi2)
    )
    return float(np.rad2deg(lam2)),float(np.rad2deg(phi2))

def aux_line(lon,lat,flow_dir,half_km=None,n=None):
    half_km=cfg.AUX_HALF_KM if half_km is None else half_km
    n=cfg.AUX_POINTS if n is None else n
    ds=np.linspace(-half_km,half_km,n)
    xs=[];ys=[]
    for s in ds:
        b=flow_dir+(np.pi/2 if s>=0 else -np.pi/2)
        x,y=forward_point(lon,lat,b,abs(float(s)))
        xs.append(x);ys.append(y)
    return np.asarray(xs),np.asarray(ys),ds

def matlab_aux_weights(n):
    k=np.arange(1,n+1,dtype=float)
    return 1.0-np.abs((n+1)/2.0-k)/(n-1)

def make_interpolators(lon,lat,u,v):
    return (
        RegularGridInterpolator(
            (lat,lon),u,
            bounds_error=False,fill_value=np.nan
        ),
        RegularGridInterpolator(
            (lat,lon),v,
            bounds_error=False,fill_value=np.nan
        )
    )

def interp_uv(ui,vi,lons,lats):
    pts=np.column_stack([lats,lons])
    return ui(pts),vi(pts)

def seed_point(lon,lat,u,v):
    ix=np.where(
        (lon>=cfg.SEED_LON_MIN)&
        (lon<=cfg.SEED_LON_MAX)
    )[0]
    iy=np.where(
        (lat>=cfg.SEED_LAT_MIN)&
        (lat<=cfg.SEED_LAT_MAX)
    )[0]

    su=u[np.ix_(iy,ix)]
    sv=v[np.ix_(iy,ix)]
    sp2=su**2+sv**2

    if not np.isfinite(sp2).any():
        raise RuntimeError("seed region invalid")

    j,i=np.unravel_index(
        np.nanargmax(sp2),sp2.shape
    )
    return (
        float(lon[ix[i]]),
        float(lat[iy[j]]),
        float(np.arctan2(sv[j,i],su[j,i]))
    )

def candidate_select(
    ui,vi,
    center_lon,center_lat,
    mean_dir,
    rescue=False,
    prev_axis_lon=None,
    prev_axis_lat=None,
    rescue_angle_deg=30.0
):
    xl,yl,_=aux_line(
        center_lon,center_lat,mean_dir
    )
    uc,vc=interp_uv(ui,vi,xl,yl)
    speed=np.hypot(uc,vc)

    valid=(
        np.isfinite(speed)&
        np.isfinite(uc)&
        np.isfinite(vc)
    )
    if not valid.any():
        return None

    if rescue and prev_axis_lon is not None:
        geom_dirs=np.array([
            angle_east_rad(
                prev_axis_lon,prev_axis_lat,x,y
            )
            for x,y in zip(xl,yl)
        ])
        geom_diff=np.abs(
            wrap_pi(geom_dirs-mean_dir)
        )
        constrained=(
            valid&
            (geom_diff<np.deg2rad(rescue_angle_deg))
        )
        if constrained.any():
            valid=constrained

    good=np.where(valid)[0]
    if not len(good):
        return None

    k=int(good[np.argmax(speed[good])])
    return {
        "lon":float(xl[k]),
        "lat":float(yl[k]),
        "speed":float(speed[k]),
    }

def track_param(
    lon,lat,u,v,
    rescue_jump_km=20.0,
    rescue_angle_deg=30.0,
    rescue_steps=20,
    loop_limit=3,
    repeat_threshold_km=1.5
):
    ui,vi=make_interpolators(
        lon,lat,u,v
    )
    lon0,lat0,theta0=seed_point(
        lon,lat,u,v
    )

    path_lon=[];path_lat=[];dirs=[]
    loop_events=0
    rescue_remaining=0
    center_lon,center_lat=lon0,lat0
    current_dir=theta0
    stop_reason="max_points"

    for _ in range(cfg.MAX_TRACK_POINTS):
        xl,yl,_=aux_line(
            center_lon,center_lat,current_dir
        )
        uu,vv=interp_uv(ui,vi,xl,yl)
        valid=np.isfinite(uu)&np.isfinite(vv)

        if not valid.any():
            stop_reason="no_valid_aux_flow"
            break

        w=matlab_aux_weights(len(xl))
        den=np.sum(w[valid])
        if den<=0:
            stop_reason="bad_aux_weight"
            break

        mean_u=np.sum(uu[valid]*w[valid])/den
        mean_v=np.sum(vv[valid]*w[valid])/den
        mean_dir=float(np.arctan2(mean_v,mean_u))

        prev_lon=path_lon[-1] if path_lon else lon0
        prev_lat=path_lat[-1] if path_lat else lat0

        cand=candidate_select(
            ui,vi,
            center_lon,center_lat,
            mean_dir,
            rescue=(rescue_remaining>0),
            prev_axis_lon=prev_lon,
            prev_axis_lat=prev_lat,
            rescue_angle_deg=rescue_angle_deg
        )

        if cand is None:
            stop_reason="no_candidate"
            break

        cl,ca=cand["lon"],cand["lat"]

        repeat_idx=None
        if len(path_lon)>8:
            hx=np.asarray(path_lon[:-5])
            hy=np.asarray(path_lat[:-5])
            dd=haversine_km(
                hx,hy,cl,ca
            )
            if (
                np.isfinite(dd).any() and
                np.nanmin(dd)<repeat_threshold_km
            ):
                repeat_idx=int(np.nanargmin(dd))

        if repeat_idx is not None:
            loop_events+=1

            path_lon=path_lon[:repeat_idx+1]
            path_lat=path_lat[:repeat_idx+1]
            dirs=dirs[:repeat_idx+1]

            if (
                loop_events>=loop_limit or
                len(path_lon)<2
            ):
                stop_reason="too_many_loops"
                break

            current_dir=dirs[-1]
            step_km=(
                cfg.STEP_EXTENSION_KM
                if path_lon[-1]>cfg.EXTENSION_LON
                else rescue_jump_km
            )
            center_lon,center_lat=forward_point(
                path_lon[-1],
                path_lat[-1],
                current_dir,
                step_km
            )
            rescue_remaining=rescue_steps
            continue

        path_lon.append(cl)
        path_lat.append(ca)
        dirs.append(mean_dir)

        if cl>=cfg.TERMINAL_LON:
            stop_reason="terminal_lon"
            break

        step_km=(
            cfg.STEP_EXTENSION_KM
            if cl>cfg.EXTENSION_LON
            else cfg.STEP_MAIN_KM
        )
        center_lon,center_lat=forward_point(
            cl,ca,mean_dir,step_km
        )
        current_dir=mean_dir
        rescue_remaining=max(
            0,rescue_remaining-1
        )

    x=np.asarray(path_lon,np.float32)
    y=np.asarray(path_lat,np.float32)

    completed=bool(
        len(x)>2 and
        np.isfinite(x).any() and
        np.nanmax(x)>=cfg.TERMINAL_LON
    )

    return {
        "lon":x,
        "lat":y,
        "direction":np.asarray(dirs,np.float32),
        "completed":completed,
        "loop_events":int(loop_events),
        "n_points":int(len(x)),
        "stop_reason":stop_reason,
        "params":{
            "rescue_jump_km":float(rescue_jump_km),
            "rescue_angle_deg":float(rescue_angle_deg),
            "rescue_steps":int(rescue_steps),
            "loop_limit":int(loop_limit),
            "repeat_threshold_km":float(repeat_threshold_km),
        }
    }

def save_track(tr,path,source_label):
    np.savez_compressed(
        path,
        lon=tr["lon"],
        lat=tr["lat"],
        direction=tr["direction"],
        completed=np.array(tr["completed"]),
        loop_events=np.array(tr["loop_events"]),
        n_points=np.array(tr["n_points"]),
        stop_reason=np.array(tr["stop_reason"]),
        tracker_version=np.array("LONG_DAILY_1993_2025_FIX6_STYLE"),
        source_label=np.array(source_label),
        params_json=np.array(
            json.dumps(tr["params"],ensure_ascii=False)
        )
    )

def load_track(path):
    with np.load(path,allow_pickle=False) as z:
        return {
            "lon":np.asarray(z["lon"],float),
            "lat":np.asarray(z["lat"],float),
            "direction":np.asarray(z["direction"],float),
            "completed":bool(z["completed"]),
            "loop_events":int(z["loop_events"]),
            "n_points":int(z["n_points"]),
            "stop_reason":str(z["stop_reason"].item()),
        }


In [ ]:

# ===== 7. 路径物理QC与failure-only retry =====
def path_basic_metrics(tr):
    x=np.asarray(tr["lon"],float)
    y=np.asarray(tr["lat"],float)

    path_km=(
        float(np.sum(
            haversine_km(
                x[:-1],y[:-1],
                x[1:],y[1:]
            )
        ))
        if len(x)>1 else 0.0
    )

    if len(x)>3:
        dirs=np.array([
            angle_east_rad(
                x[i],y[i],
                x[i+1],y[i+1]
            )
            for i in range(len(x)-1)
        ])
        turns=np.abs(
            np.rad2deg(
                wrap_pi(np.diff(dirs))
            )
        )
        p95=float(
            np.nanpercentile(turns,95)
        )
    else:
        p95=np.nan

    return {
        "path_km":path_km,
        "p95_turn_deg":p95,
        "max_lon":float(np.nanmax(x)) if len(x) else np.nan,
        "final_lon":float(x[-1]) if len(x) else np.nan,
        "final_lat":float(y[-1]) if len(y) else np.nan,
    }

def ridge_adherence(
    lon_grid,lat_grid,u,v,tr
):
    x=np.asarray(tr["lon"],float)
    y=np.asarray(tr["lat"],float)

    if len(x)<5:
        return np.nan,np.nan,np.nan

    ui,vi=make_interpolators(
        lon_grid,lat_grid,u,v
    )

    ids=np.arange(
        2,len(x)-2,
        cfg.RIDGE_SAMPLE_STRIDE
    )
    ratios=[]

    for i in ids:
        tangent=angle_east_rad(
            x[i-1],y[i-1],
            x[i+1],y[i+1]
        )
        xl,yl,_=aux_line(
            x[i],y[i],tangent,
            half_km=cfg.RIDGE_HALF_KM,
            n=cfg.RIDGE_POINTS
        )
        uu,vv=interp_uv(
            ui,vi,xl,yl
        )
        sp=np.hypot(uu,vv)
        good=np.isfinite(sp)

        if not good.any():
            continue

        uc,vc=interp_uv(
            ui,vi,[x[i]],[y[i]]
        )
        center_speed=float(
            np.hypot(uc[0],vc[0])
        )
        mx=float(np.nanmax(sp[good]))

        if np.isfinite(center_speed) and mx>1e-8:
            ratios.append(
                min(center_speed/mx,1.25)
            )

    if not ratios:
        return np.nan,np.nan,np.nan

    rr=np.asarray(ratios,float)

    return (
        float(np.nanmedian(rr)),
        float(np.nanpercentile(rr,10)),
        float(np.mean(rr<0.75))
    )

def track_quality(field,tr):
    b=path_basic_metrics(tr)
    rmed,rp10,rlow=ridge_adherence(
        field["lon"],field["lat"],
        field["u"],field["v"],
        tr
    )
    return {
        **b,
        "ridge_median":rmed,
        "ridge_p10":rp10,
        "ridge_low_frac":rlow,
    }

def physical_gate(m):
    return bool(
        np.isfinite(m["ridge_median"]) and
        np.isfinite(m["ridge_p10"]) and
        m["ridge_median"]>=cfg.MIN_RIDGE_MEDIAN and
        m["ridge_p10"]>=cfg.MIN_RIDGE_P10 and
        m["ridge_low_frac"]<=cfg.MAX_RIDGE_LOW_FRAC and
        (
            not np.isfinite(m["p95_turn_deg"]) or
            m["p95_turn_deg"]<=cfg.MAX_P95_TURN_DEG
        )
    )

def resample_track(tr,n=160):
    x=np.asarray(tr["lon"],float)
    y=np.asarray(tr["lat"],float)
    if len(x)<2:
        return x,y

    seg=haversine_km(
        x[:-1],y[:-1],
        x[1:],y[1:]
    )
    s=np.r_[0,np.cumsum(seg)]
    keep=np.r_[True,np.diff(s)>1e-6]
    x,y,s=x[keep],y[keep],s[keep]

    if len(x)<2 or s[-1]<=0:
        return x,y

    ss=np.linspace(0,s[-1],n)
    return (
        np.interp(ss,s,x),
        np.interp(ss,s,y)
    )

def symmetric_path_distance_km(
    tr1,tr2,n=120
):
    if tr1 is None or tr2 is None:
        return np.nan

    x1,y1=resample_track(tr1,n=n)
    x2,y2=resample_track(tr2,n=n)

    if len(x1)==0 or len(x2)==0:
        return np.nan

    lat0=float(
        np.nanmean(np.r_[y1,y2])
    )
    lon0=float(
        np.nanmean(np.r_[x1,x2])
    )
    c=np.cos(np.deg2rad(lat0))

    A=np.column_stack([
        (x1-lon0)*111.32*c,
        (y1-lat0)*111.32
    ])
    B=np.column_stack([
        (x2-lon0)*111.32*c,
        (y2-lat0)*111.32
    ])

    d=np.sqrt(
        ((A[:,None,:]-B[None,:,:])**2)
        .sum(axis=2)
    )
    return float(
        np.median(
            np.r_[d.min(axis=1),d.min(axis=0)]
        )
    )

def run_failure_only_retry(
    field,
    prev_track=None
):
    candidates=[]

    for jump,ang,loop_lim,rpt in cfg.RETRY_GRID:
        tr=track_param(
            field["lon"],field["lat"],
            field["u"],field["v"],
            rescue_jump_km=jump,
            rescue_angle_deg=ang,
            rescue_steps=cfg.RESCUE_STEPS,
            loop_limit=loop_lim,
            repeat_threshold_km=rpt
        )

        if not tr["completed"]:
            continue

        q=track_quality(field,tr)

        rec={
            "track":tr,
            "quality":q,
            "physical_gate":physical_gate(q),
            "temporal_prev_km":(
                symmetric_path_distance_km(
                    tr,prev_track
                )
                if prev_track is not None
                else np.nan
            )
        }
        candidates.append(rec)

        if sum(c["physical_gate"] for c in candidates) >= cfg.MAX_GOOD_RETRY_CANDIDATES:
            break

    if not candidates:
        return None,[]

    good=[
        c for c in candidates
        if c["physical_gate"]
    ]

    pool=good if good else candidates

    if prev_track is not None:
        pool=sorted(
            pool,
            key=lambda c:(
                0 if c["physical_gate"] else 1,
                c["temporal_prev_km"]
                if np.isfinite(c["temporal_prev_km"])
                else 1e9,
                -c["quality"]["ridge_median"]
                if np.isfinite(c["quality"]["ridge_median"])
                else 1e9,
            )
        )
    else:
        pool=sorted(
            pool,
            key=lambda c:(
                0 if c["physical_gate"] else 1,
                -c["quality"]["ridge_median"]
                if np.isfinite(c["quality"]["ridge_median"])
                else 1e9,
                c["quality"]["p95_turn_deg"]
                if np.isfinite(c["quality"]["p95_turn_deg"])
                else 1e9,
            )
        )

    return pool[0],candidates


In [ ]:

# ===== 8. 单日处理函数：baseline-first，真正失败/可疑时才retry =====
def meta_path(dt):
    return DIRS["track_meta"]/f"{pd.Timestamp(dt):%Y%m%d}.json"

def auto_track_path(dt):
    return DIRS["tracks_auto"]/f"{pd.Timestamp(dt):%Y%m%d}.npz"

def manual_cache_path(dt):
    return DIRS["manual_cache"]/f"{pd.Timestamp(dt):%Y%m%d}.npz"

def write_meta(dt,obj):
    meta_path(dt).write_text(
        json.dumps(
            obj,indent=2,
            ensure_ascii=False,
            default=str
        ),
        encoding="utf-8"
    )

def process_one_day(row,prev_track=None):
    dt=pd.Timestamp(row.date)

    field=read_50m_field(
        row.selected_path,
        dt,
        row.selected_time_index
    )

    baseline=track_param(
        field["lon"],field["lat"],
        field["u"],field["v"],
        rescue_jump_km=cfg.BASE_RESCUE_JUMP_KM,
        rescue_angle_deg=cfg.BASE_RESCUE_ANGLE_DEG,
        rescue_steps=cfg.RESCUE_STEPS,
        loop_limit=cfg.BASE_LOOP_LIMIT,
        repeat_threshold_km=cfg.BASE_REPEAT_THRESHOLD_KM
    )

    baseq=track_quality(
        field,baseline
    ) if baseline["n_points"] else {
        "path_km":np.nan,
        "p95_turn_deg":np.nan,
        "max_lon":np.nan,
        "final_lon":np.nan,
        "final_lat":np.nan,
        "ridge_median":np.nan,
        "ridge_p10":np.nan,
        "ridge_low_frac":np.nan,
    }

    base_good=(
        baseline["completed"] and
        physical_gate(baseq)
    )

    chosen=baseline
    chosenq=baseq
    source="baseline"
    retry_trials=0

    # 只有baseline未完成或物理质量明显可疑才自动重试
    if not base_good:
        best,cands=run_failure_only_retry(
            field,prev_track
        )
        retry_trials=len(cands)

        if (
            best is not None and
            best["track"]["completed"] and
            best["physical_gate"]
        ):
            chosen=best["track"]
            chosenq=best["quality"]
            source="retry"

    auto_good=(
        chosen["completed"] and
        physical_gate(chosenq)
    )

    # 与最近前一条accepted track只做软诊断
    prev_dist=(
        symmetric_path_distance_km(
            chosen,prev_track
        )
        if (
            prev_track is not None and
            chosen["n_points"]>1
        )
        else np.nan
    )

    reasons=[]

    if not chosen["completed"]:
        reasons.append(
            "incomplete:"+chosen["stop_reason"]
        )

    if chosen["completed"] and not physical_gate(chosenq):
        reasons.append("physical_qc_failed")

    # 这里只初步标记极端前日跳变；后面还会做双向时间审计
    if np.isfinite(prev_dist) and prev_dist>cfg.TEMPORAL_WARN_KM_1D*1.8:
        reasons.append(
            f"extreme_prev_jump_{prev_dist:.1f}km"
        )

    status=(
        "accepted_auto"
        if auto_good and not reasons
        else "manual_required"
    )

    save_track(
        chosen,
        auto_track_path(dt),
        source
    )

    if (
        cfg.SAVE_FIELD_FOR_ALL_DAYS or
        (
            cfg.SAVE_FIELD_FOR_MANUAL_ONLY and
            status=="manual_required"
        )
    ):
        save_field_npz(
            field,
            manual_cache_path(dt)
        )

    meta={
        "date":str(dt.date()),
        "selected_source":row.selected_source,
        "selected_path":row.selected_path,
        "status":status,
        "auto_source":source,
        "reasons":reasons,
        "baseline_completed":bool(baseline["completed"]),
        "baseline_stop_reason":baseline["stop_reason"],
        "baseline_loop_events":int(baseline["loop_events"]),
        "retry_trials":int(retry_trials),
        "prev_path_distance_km":(
            None if not np.isfinite(prev_dist)
            else float(prev_dist)
        ),
        **{
            k:(
                None if not np.isfinite(v)
                else float(v)
            )
            for k,v in chosenq.items()
        }
    }
    write_meta(dt,meta)

    return meta,chosen,field


In [ ]:

# ===== 9. 跨年代 smoke test：正式批处理前必须先通过 =====
smoke_rows=[]

for ds in cfg.SMOKE_DATES:
    dt=pd.Timestamp(ds)
    q=manifest[
        (manifest.date==dt)&
        (manifest.raw_status=="present")
    ]
    if not len(q):
        smoke_rows.append({
            "date":dt,
            "status":"missing_raw"
        })
        continue

    r=q.iloc[0]

    try:
        field=read_50m_field(
            r.selected_path,
            dt,
            r.selected_time_index
        )
        tr=track_param(
            field["lon"],field["lat"],
            field["u"],field["v"],
            rescue_jump_km=cfg.BASE_RESCUE_JUMP_KM,
            rescue_angle_deg=cfg.BASE_RESCUE_ANGLE_DEG,
            rescue_steps=cfg.RESCUE_STEPS,
            loop_limit=cfg.BASE_LOOP_LIMIT,
            repeat_threshold_km=cfg.BASE_REPEAT_THRESHOLD_KM
        )
        ql=track_quality(field,tr)

        smoke_rows.append({
            "date":dt,
            "source":r.selected_source,
            "completed":tr["completed"],
            "stop_reason":tr["stop_reason"],
            **ql
        })
    except Exception as e:
        smoke_rows.append({
            "date":dt,
            "status":"error",
            "error":repr(e)
        })

smoke_df=pd.DataFrame(smoke_rows)
smoke_df.to_csv(
    DIRS["reports"]/"04_跨年代Tracker_SmokeTest.csv",
    index=False,encoding="utf-8-sig"
)
display(smoke_df)

if "status" in smoke_df and (smoke_df["status"]=="error").any():
    raise RuntimeError(
        "Smoke test 出现读取/追踪错误。"
        "不要直接开始32年全量批处理。"
    )

if cfg.SMOKE_TEST_ONLY:
    raise SystemExit(
        "SMOKE_TEST_ONLY=True：已完成预检，未进入全量批处理。"
    )


In [ ]:

# ===== 10. 全量1993–2025自动处理，可断点续跑 =====
present=manifest[
    manifest.raw_status=="present"
].sort_values("date").copy()

rows=[]
last_good_date=None
last_good_track=None

# 先用已有meta恢复最近accepted轨迹，以支持断点续跑
existing_meta=[]
for mp in DIRS["track_meta"].glob("*.json"):
    try:
        m=json.loads(mp.read_text(encoding="utf-8"))
        existing_meta.append(m)
    except Exception:
        pass

existing_df=pd.DataFrame(existing_meta)
if len(existing_df):
    existing_df["date"]=pd.to_datetime(existing_df["date"])
    accepted_old=existing_df[
        existing_df["status"]=="accepted_auto"
    ].sort_values("date")
else:
    accepted_old=pd.DataFrame()

for ii,row in enumerate(
    tqdm(
        present.itertuples(index=False),
        total=len(present),
        desc="1993–2025 auto axis"
    ),
    start=1
):
    dt=pd.Timestamp(row.date)
    mp=meta_path(dt)
    tp=auto_track_path(dt)

    # 找当前日期之前最近的accepted track
    if (
        last_good_date is not None and
        (dt-last_good_date).days>cfg.TEMPORAL_MAX_GAP_DAYS
    ):
        last_good_date=None
        last_good_track=None

    if (
        mp.exists() and
        tp.exists() and
        not cfg.FORCE_REPROCESS
    ):
        try:
            m=json.loads(
                mp.read_text(encoding="utf-8")
            )
            tr=load_track(tp)

            rows.append(m)

            if (
                m.get("status")=="accepted_auto" and
                tr["completed"]
            ):
                last_good_date=dt
                last_good_track=tr

            continue
        except Exception:
            pass

    try:
        m,tr,field=process_one_day(
            row,
            prev_track=last_good_track
        )
        rows.append(m)

        if (
            m["status"]=="accepted_auto" and
            tr["completed"]
        ):
            last_good_date=dt
            last_good_track=tr

    except Exception as e:
        m={
            "date":str(dt.date()),
            "selected_source":row.selected_source,
            "selected_path":row.selected_path,
            "status":"manual_required",
            "auto_source":"error",
            "reasons":[
                "processing_error:"+repr(e)
            ]
        }
        write_meta(dt,m)
        rows.append(m)

    if ii%cfg.FLUSH_REPORT_EVERY==0:
        pd.DataFrame(rows).to_csv(
            DIRS["reports"]/"05_自动追踪运行中间报告.csv",
            index=False,encoding="utf-8-sig"
        )
        gc.collect()

auto_report=pd.DataFrame(rows)
auto_report["date"]=pd.to_datetime(
    auto_report["date"]
)
auto_report=auto_report.sort_values("date")

auto_report.to_csv(
    DIRS["reports"]/"05_自动追踪结果.csv",
    index=False,encoding="utf-8-sig"
)

print(auto_report["status"].value_counts())
print(
    "auto accepted rate:",
    (auto_report.status=="accepted_auto").mean()
)


In [ ]:

# ===== 11. 双向时间连续性审计：筛出“完成但可疑”的路径 =====
accepted=auto_report[
    auto_report.status=="accepted_auto"
].copy()

accepted_map={}
for dt in accepted["date"]:
    p=auto_track_path(dt)
    if p.exists():
        accepted_map[pd.Timestamp(dt)]=load_track(p)

temporal_rows=[]
accepted_dates=sorted(accepted_map)

for i,dt in enumerate(
    tqdm(
        accepted_dates,
        desc="temporal continuity audit"
    )
):
    tr=accepted_map[dt]

    prev_dt=None
    next_dt=None

    for j in range(i-1,max(-1,i-4),-1):
        cand=accepted_dates[j]
        if (dt-cand).days<=cfg.TEMPORAL_MAX_GAP_DAYS:
            prev_dt=cand
            break

    for j in range(i+1,min(len(accepted_dates),i+4)):
        cand=accepted_dates[j]
        if (cand-dt).days<=cfg.TEMPORAL_MAX_GAP_DAYS:
            next_dt=cand
            break

    pdist=(
        symmetric_path_distance_km(
            tr,accepted_map[prev_dt]
        )
        if prev_dt is not None else np.nan
    )
    ndist=(
        symmetric_path_distance_km(
            tr,accepted_map[next_dt]
        )
        if next_dt is not None else np.nan
    )

    p_gap=(
        (dt-prev_dt).days
        if prev_dt is not None else np.nan
    )
    n_gap=(
        (next_dt-dt).days
        if next_dt is not None else np.nan
    )

    temporal_rows.append({
        "date":dt,
        "prev_date":prev_dt,
        "prev_gap_days":p_gap,
        "prev_distance_km":pdist,
        "next_date":next_dt,
        "next_gap_days":n_gap,
        "next_distance_km":ndist,
    })

temporal=pd.DataFrame(temporal_rows)

def temporal_limit(gap):
    if not np.isfinite(gap):
        return np.nan
    gap=float(gap)
    if gap<=1:
        return cfg.TEMPORAL_WARN_KM_1D
    return (
        cfg.TEMPORAL_WARN_KM_1D+
        (cfg.TEMPORAL_WARN_KM_3D-cfg.TEMPORAL_WARN_KM_1D)*
        (gap-1)/(cfg.TEMPORAL_MAX_GAP_DAYS-1)
    )

def suspicious_temporal(row):
    bad_prev=(
        np.isfinite(row.prev_distance_km) and
        row.prev_distance_km>temporal_limit(row.prev_gap_days)
    )
    bad_next=(
        np.isfinite(row.next_distance_km) and
        row.next_distance_km>temporal_limit(row.next_gap_days)
    )

    # 必须至少一侧极端，或双侧都异常
    extreme_prev=(
        np.isfinite(row.prev_distance_km) and
        row.prev_distance_km>1.35*temporal_limit(row.prev_gap_days)
    )
    extreme_next=(
        np.isfinite(row.next_distance_km) and
        row.next_distance_km>1.35*temporal_limit(row.next_gap_days)
    )

    return bool(
        (bad_prev and bad_next) or
        extreme_prev or
        extreme_next
    )

temporal["temporal_suspicious"]=temporal.apply(
    suspicious_temporal,axis=1
)

temporal.to_csv(
    DIRS["reports"]/"06_相邻日路径连续性审计.csv",
    index=False,encoding="utf-8-sig"
)

print(
    "temporal suspicious:",
    int(temporal.temporal_suspicious.sum())
)


In [ ]:

# ===== 12. 构建最终人工复核队列，并只缓存这些天的50m场 =====
base_manual=auto_report[
    auto_report.status=="manual_required"
][["date","status","reasons"]].copy()

temp_manual=temporal[
    temporal.temporal_suspicious
][[
    "date",
    "prev_distance_km",
    "next_distance_km"
]].copy()

temp_manual["status"]="manual_required"
temp_manual["reasons"]=temp_manual.apply(
    lambda r:[
        f"temporal_suspicious_prev={r.prev_distance_km:.1f}km",
        f"next={r.next_distance_km:.1f}km"
    ],
    axis=1
)

manual_queue=pd.concat(
    [
        base_manual[["date","status","reasons"]],
        temp_manual[["date","status","reasons"]]
    ],
    ignore_index=True
)

if len(manual_queue):
    manual_queue=(
        manual_queue
        .sort_values("date")
        .drop_duplicates("date",keep="first")
        .reset_index(drop=True)
    )

manual_queue.to_csv(
    DIRS["reports"]/"07_人工复核队列.csv",
    index=False,encoding="utf-8-sig"
)

# 给人工队列补齐50m cache
for dt in tqdm(
    manual_queue["date"].tolist()
    if len(manual_queue) else [],
    desc="cache manual fields"
):
    dt=pd.Timestamp(dt)
    cp=manual_cache_path(dt)
    if cp.exists():
        continue

    r=manifest[
        manifest.date==dt
    ].iloc[0]

    try:
        field=read_50m_field(
            r.selected_path,
            dt,
            r.selected_time_index
        )
        save_field_npz(
            field,cp
        )
    except Exception as e:
        print(
            "[manual cache error]",
            dt.date(),repr(e)
        )

print(
    "manual queue:",
    len(manual_queue),
    "/ present days:",
    len(present),
    f"({len(manual_queue)/max(len(present),1):.2%})"
)

if len(manual_queue)/max(len(present),1)>0.05:
    warnings.warn(
        "人工复核比例超过5%。"
        "建议先抽查manual_queue是否阈值过严，再开始大规模人工标注。"
    )

display(manual_queue.head(30))


In [ ]:

# ===== 13. 2024→2025 产品切换边界专项审计 =====
transition_dates=pd.date_range(
    "2024-12-20",
    "2025-01-10",
    freq="D"
)

rows=[]

for dt in transition_dates:
    q=temporal[
        temporal.date==dt
    ]
    m=auto_report[
        auto_report.date==dt
    ]

    if not len(m):
        continue

    rows.append({
        "date":dt,
        "source":m.iloc[0]["selected_source"],
        "status":m.iloc[0]["status"],
        "prev_distance_km":(
            q.iloc[0]["prev_distance_km"]
            if len(q) else np.nan
        ),
        "next_distance_km":(
            q.iloc[0]["next_distance_km"]
            if len(q) else np.nan
        ),
    })

transition=pd.DataFrame(rows)
transition.to_csv(
    DIRS["reports"]/"08_2024_2025产品切换路径连续性审计.csv",
    index=False,encoding="utf-8-sig"
)
display(transition)


In [ ]:

# ===== 14. 数据构建阶段 Handoff =====
handoff={
    "project_root":str(ROOT),
    "multiyear_root":str(MY_ROOT),
    "recent_root":str(RECENT_ROOT),
    "date_start":str(manifest.date.min().date()),
    "date_end":str(manifest.date.max().date()),
    "theoretical_days":int(len(manifest)),
    "present_raw_days":int((manifest.raw_status=="present").sum()),
    "missing_raw_days":int((manifest.raw_status=="missing").sum()),
    "auto_processed_days":int(len(auto_report)),
    "auto_accepted_days":int((auto_report.status=="accepted_auto").sum()),
    "manual_review_days":int(len(manual_queue)),
    "selected_source_counts":(
        manifest["selected_source"]
        .value_counts()
        .to_dict()
    ),
    "important_outputs":{
        "manifest":str(manifest_path),
        "auto_report":str(DIRS["reports"]/"05_自动追踪结果.csv"),
        "manual_queue":str(DIRS["reports"]/"07_人工复核队列.csv"),
        "manual_cache":str(DIRS["manual_cache"]),
        "auto_tracks":str(DIRS["tracks_auto"]),
    }
}

(DIRS["reports"]/"DATA_BUILD_HANDOFF.json").write_text(
    json.dumps(
        handoff,indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print(json.dumps(
    handoff,indent=2,
    ensure_ascii=False
))
print(
    "\n下一步：打开配套 "
    "Kuroshio_Daily_1993_2025_ManualQC_Interactive_VSCode_WSL.ipynb"
)
